# K-Nearest Neighbors Regression

This notebook demonstrates KNN regression on a small one-dimensional dataset and on the California Housing dataset. It also shows why feature scaling and quantitative evaluation matter for distance-based models.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## 1. One-dimensional example

KNN regression predicts the average target value of the nearest training samples. The prediction curve is piecewise rather than a fitted straight line.


In [ ]:
X_small = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
y_small = np.array([1.0, 1.8, 3.2, 3.7, 5.1])

small_model = KNeighborsRegressor(n_neighbors=3)
small_model.fit(X_small, y_small)

x_grid = np.linspace(1, 5, 300).reshape(-1, 1)
y_grid = small_model.predict(x_grid)
X_new = np.array([[2.6]])
y_new = small_model.predict(X_new)

plt.figure(figsize=(9, 5))
plt.scatter(X_small[:, 0], y_small, color="tab:blue", s=60, label="Training data")
plt.plot(x_grid[:, 0], y_grid, color="tab:orange", label="KNN prediction (k=3)")
plt.scatter(X_new[:, 0], y_new, color="tab:red", marker="x", s=100, label=f"Prediction: {y_new[0]:.2f}")
plt.xlabel("Feature")
plt.ylabel("Target")
plt.title("KNN Regression on a Small Dataset")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 2. California Housing

The data is split before preprocessing. `StandardScaler` is fitted only on training features through a pipeline, preventing test-set leakage. Uniform and distance weighting are compared with MAE and R².


In [ ]:
housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Uniform weights": make_pipeline(
        StandardScaler(), KNeighborsRegressor(n_neighbors=5, weights="uniform")
    ),
    "Distance weights": make_pipeline(
        StandardScaler(), KNeighborsRegressor(n_neighbors=5, weights="distance")
    ),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    print(
        f"{name}: "
        f"MAE={mean_absolute_error(y_test, predictions[name]):.3f}, "
        f"R^2={r2_score(y_test, predictions[name]):.3f}"
    )

limits = [min(y_test.min(), *(p.min() for p in predictions.values())),
          max(y_test.max(), *(p.max() for p in predictions.values()))]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (name, predicted) in zip(axes, predictions.items()):
    ax.scatter(y_test, predicted, alpha=0.25, s=14, color="tab:orange")
    ax.plot(limits, limits, "--", color="tab:blue", label="Ideal prediction")
    ax.set_title(name)
    ax.set_xlabel("True house value ($100,000s)")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("Predicted house value ($100,000s)")
fig.suptitle("KNN Regression: True vs. Predicted Values")
plt.tight_layout()
plt.show()


## Takeaways

- KNN depends on distances, so features must be placed on comparable scales.
- Smaller MAE and larger R² indicate better test performance.
- Distance weighting can help when nearby observations should contribute more strongly than distant neighbors.
